Step 0 — Mount Drive and Setup

Reused from Week 3 — no changes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/doctalk'
STANDALONE_DIR = f'{PROJECT_DIR}/faiss_index/standalone'
DOCS_DIR = f'{PROJECT_DIR}/uploaded_docs'

os.makedirs(STANDALONE_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

print('Google Drive mounted and project folders ready.')
print(f'Project folder: {PROJECT_DIR}')

Step 1 — Install Dependencies

Reused from Week 3 — added pdfplumber for table extraction.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q pymupdf
!pip install -q sentence-transformers
!pip install -q groq langchain-groq
!pip install -q python-docx docx2txt
!pip install -q pdfplumber

print('All dependencies installed.')

Step 2 — Load Embedding Model and LLM

Reused from Week 3 — no changes.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from google.colab import userdata

print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print('Embedding model loaded.')

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name='llama-3.3-70b-versatile',
    temperature=0
)
print('Groq LLM loaded.')

Step 3 — Extract Tables with pdfplumber

New in Week 4 — extracts tables from PDFs as structured text.

In [ ]:
import pdfplumber

def extract_tables(filepath):
    tables_text = []

    with pdfplumber.open(filepath) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table in tables:
                if not table:
                    continue

                # Convert table rows to readable text
                rows = []
                for row in table:
                    # Clean None values and join cells
                    cleaned = [cell.strip() if cell else '' for cell in row]
                    rows.append(' | '.join(cleaned))

                table_text = '\n'.join(rows)
                tables_text.append({
                    'page': page_num,
                    'content': table_text
                })

    print(f'Extracted {len(tables_text)} table(s) from {os.path.basename(filepath)}')
    return tables_text

print('Table extractor ready.')

Step 4 — Upload and Test Table Extraction

New in Week 4 — upload a PDF with tables and verify extraction.

In [ ]:
from google.colab import files
import shutil

print('Upload a PDF with tables...')
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f'{DOCS_DIR}/{filename}'
    shutil.copy(filename, dest)
    PDF_PATH = dest
    print(f'Saved: {dest}')

tables = extract_tables(PDF_PATH)

print(f'\n--- Extracted Tables ---')
for i, table in enumerate(tables):
    print(f'\nTable {i+1} (Page {table["page"] + 1}):')
    print(table['content'])
    print()

Step 5 — Convert Tables to LangChain Documents

New in Week 4 — wraps extracted tables as Document objects for indexing.

In [ ]:
from langchain_core.documents import Document

def tables_to_documents(tables, source_path):
    docs = []
    for table in tables:
        doc = Document(
            page_content=table['content'],
            metadata={
                'source': source_path,
                'page': table['page'],
                'type': 'table'
            }
        )
        docs.append(doc)

    print(f'Created {len(docs)} table document(s)')
    return docs

table_docs = tables_to_documents(tables, PDF_PATH)

print(f'\n--- Example Table Document ---')
print(f'Content: {table_docs[0].page_content}')
print(f'Metadata: {table_docs[0].metadata}')

Step 6 — Updated Document Loader with Table Support

Updated from Week 3 — now extracts both text and tables from PDFs.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_document_with_tables(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == '.pdf':
        # Load regular text
        loader = PyMuPDFLoader(filepath)
        docs = loader.load()

        # Extract and append tables
        tables = extract_tables(filepath)
        if tables:
            table_docs = tables_to_documents(tables, filepath)
            docs.extend(table_docs)
            print(f'Added {len(table_docs)} table chunk(s) to document')

    elif ext == '.docx':
        loader = Docx2txtLoader(filepath)
        docs = loader.load()
    elif ext == '.txt':
        loader = TextLoader(filepath)
        docs = loader.load()
    else:
        raise ValueError(f'Unsupported file type: {ext}. Supported types: PDF, DOCX, TXT')

    if not docs:
        raise ValueError(f'Document is empty or could not be parsed: {filepath}')

    total_text = ''.join([d.page_content for d in docs]).strip()
    if len(total_text) < 100:
        raise ValueError(f'Document contains too little text to be useful: {filepath}')

    print(f'Total: {len(docs)} document chunk(s) loaded from {os.path.basename(filepath)}')
    return docs

def chunk_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50,
        length_function=len
    )

    text_docs = [d for d in docs if d.metadata.get('type') != 'table']
    table_docs = [d for d in docs if d.metadata.get('type') == 'table']

    chunks = splitter.split_documents(text_docs)
    chunks.extend(table_docs)  # add table docs as-is without splitting

    print(f'Split into {len(chunks)} chunks ({len(table_docs)} table chunks kept intact)')
    return chunks

Step 7 — Build FAISS Index

Reused from Week 3 — no changes.

In [ ]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm

def build_index(chunks, index_path):
    os.makedirs(index_path, exist_ok=True)

    batch_size = 50
    all_embeddings = []

    for i in tqdm(range(0, len(chunks), batch_size), desc='Embedding chunks', unit='batch'):
        batch = chunks[i:i + batch_size]
        batch_texts = [chunk.page_content for chunk in batch]
        batch_embeddings = embeddings.embed_documents(batch_texts)
        all_embeddings.extend(batch_embeddings)

    texts = [chunk.page_content for chunk in chunks]
    metadatas = [chunk.metadata for chunk in chunks]

    vectorstore = FAISS.from_embeddings(
        list(zip(texts, all_embeddings)),
        embeddings,
        metadatas=metadatas
    )

    vectorstore.save_local(index_path)
    print(f'Index saved: {index_path}')
    return vectorstore

print('Index builder ready.')

Step 8 — Index Document with Tables

New in Week 4 — indexes the uploaded PDF including its tables.

In [ ]:
doc_name = os.path.splitext(os.path.basename(PDF_PATH))[0]
index_path = f'{STANDALONE_DIR}/{doc_name}'

docs = load_document_with_tables(PDF_PATH)
chunks = chunk_documents(docs)
vectorstore = build_index(chunks, index_path)

print(f'\nDocument indexed: {doc_name}')

Step 9 — Build RAG Chain and Query

Reused from Week 3 — test that table data is retrievable.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

template = """
You are a helpful assistant that answers questions based strictly on the provided context.
If the answer is not found in the context, say "I could not find an answer in the document."
Do not use any knowledge outside of the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=['context', 'question'],
    template=template
)

def format_chunks(docs):
    return '\n\n'.join([
        f'[Page {doc.metadata.get("page", 0) + 1}] [Type: {doc.metadata.get("type", "text")}]\n{doc.page_content}'
        for doc in docs
    ])

def safe_ask(question, rag_chain):
    if not question.strip():
        print('Please enter a valid question.')
        return None
    answer = rag_chain.invoke(question)
    return answer

retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

rag_chain = (
    {
        'context': RunnableLambda(lambda q: retriever.invoke(q)) | format_chunks,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain built.')

Step 10 — Test Table Query

New in Week 4 — verify that table data is retrieved correctly.

In [ ]:
question = "" # enter a question that relates to table data in your document

answer = safe_ask(question, rag_chain)

if answer is None:
    pass
else:
    source_docs = retriever.invoke(question)
    print(f'Question: {question}')
    print(f'\nAnswer: {answer}')
    print(f'\nSources:')
    for i, doc in enumerate(source_docs[:3]):
        doc_type = doc.metadata.get('type', 'text')
        print(f'  [{i+1}] Page {doc.metadata.get("page", 0) + 1} [{doc_type}]: {doc.page_content[:120]}...')